Step 0 - Install Streamlit

In [1]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 76.3 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
import zipfile

# --- 1. CONFIGURATION & PAGE SETUP ---
st.set_page_config(
    page_title="Campus Service Queue Prioritization",
    page_icon="📋",
    layout="wide"
)

st.title("Campus Service Queue Prioritization")
st.write("Manage campus service requests using Priority Scoring and FIFO.")

# --- 2. PRIORITY SCORING & RANKING FUNCTIONS ---
urgency_points = {
    "High": 40,
    "Medium": 25,
    "Low": 10
}

def score_records(data):
    df = data.copy()
    df["arrival_time"] = pd.to_datetime(df["arrival_time"], errors="coerce")
    df["estimated_duration"] = pd.to_numeric(df["estimated_duration"], errors="coerce")

    df["urgency"] = df["urgency"].fillna("Low")
    df["service_type"] = df["service_type"].fillna("General Inquiry")
    df["student_type"] = df["student_type"].fillna("Non-Graduating")
    df["document_complete"] = df["document_complete"].fillna("No")
    df["special_assistance"] = df["special_assistance"].fillna("No")

    df["estimated_duration"] = df["estimated_duration"].fillna(
        df["estimated_duration"].median() if not df["estimated_duration"].dropna().empty else 15
    )

    df["priority_score"] = (
        df["urgency"].map(urgency_points).fillna(0)
        + np.where(df["student_type"] == "Graduating", 25, 10)
        + np.where(df["special_assistance"] == "Yes", 20, 0)
        + np.where(df["document_complete"] == "No", -10, 0)
        + np.where(df["estimated_duration"] <= 15, 5, 0)
    )
    return df

def rank_records(data):
    priority_order = data.sort_values(
        ["priority_score", "arrival_time", "record_id"],
        ascending=[False, True, True],
        na_position="last"
    ).reset_index(drop=True)

    priority_order.insert(0, "priority_rank", range(1, len(priority_order) + 1))

    fifo_order = data.sort_values(
        ["arrival_time", "record_id"],
        ascending=[True, True],
        na_position="last"
    ).reset_index(drop=True)

    fifo_order.insert(0, "fifo_rank", range(1, len(fifo_order) + 1))

    comparison = priority_order[["record_id", "priority_rank", "priority_score"]].merge(
        fifo_order[["record_id", "fifo_rank"]],
        on="record_id"
    )
    comparison["rank_difference"] = comparison["fifo_rank"] - comparison["priority_rank"]
    return priority_order, fifo_order, comparison

# --- 3. DATASET UPLOAD & VALIDATION ---
st.header("1. Upload Your Datasets")

original_file = st.file_uploader(
    "Upload the original complete dataset (CSV)",
    type=["csv"],
    key="original"
)

missing_file = st.file_uploader(
    "Upload the dataset containing missing values (CSV)",
    type=["csv"],
    key="missing"
)

required_columns = [
    "record_id", "arrival_time", "service_type", "urgency",
    "estimated_duration", "student_type", "document_complete", "special_assistance"
]

original = None
missing = None

if original_file is not None:
    original = pd.read_csv(original_file)
    absent = [col for col in required_columns if col not in original.columns]
    if absent:
        st.error(f"Original dataset is missing columns: {absent}")
        original = None
    else:
        st.success(f"Original dataset loaded: {len(original)} records")

if missing_file is not None:
    missing = pd.read_csv(missing_file)
    absent = [col for col in required_columns if col not in missing.columns]
    if absent:
        st.error(f"Missing-value dataset is missing columns: {absent}")
        missing = None
    else:
        st.success(f"Missing-value dataset loaded: {len(missing)} records")

# --- 4. MANUAL SERVICE REQUEST FORM ---
st.header("2. Add a Service Request")

with st.form("service_form"):
    col1, col2 = st.columns(2)
    with col1:
        arrival_date = st.date_input("Arrival Date")
        arrival_clock = st.time_input("Arrival Time")
        service_type = st.selectbox(
            "Service Type",
            ["Enrollment", "Document Request", "Tuition Payment", "Clearance", "Academic Advising", "General Inquiry", "Other"]
        )
        urgency = st.selectbox("Urgency", ["High", "Medium", "Low"])
    with col2:
        estimated_duration = st.number_input(
            "Estimated Service Duration (minutes)",
            min_value=1, max_value=480, value=15, step=1
        )
        student_type = st.selectbox("Student Type", ["Graduating", "Non-Graduating"])
        document_complete = st.selectbox("Are the required documents complete?", ["Yes", "No"])
        special_assistance = st.selectbox("Does the student need special assistance?", ["No", "Yes"])

    submitted = st.form_submit_button("Add Service Request", use_container_width=True)

if "manual_records" not in st.session_state:
    st.session_state.manual_records = []

if submitted:
    existing_ids = []
    if original is not None and "record_id" in original.columns:
        existing_ids = original["record_id"].dropna().tolist()
    existing_ids += [r["record_id"] for r in st.session_state.manual_records]

    numeric_ids = pd.to_numeric(pd.Series(existing_ids), errors="coerce")
    next_id = int(numeric_ids.max()) + 1 if numeric_ids.notna().any() else 1
    arrival = pd.Timestamp.combine(arrival_date, arrival_clock)

    new_record = {
        "record_id": next_id,
        "arrival_time": arrival,
        "service_type": service_type,
        "urgency": urgency,
        "estimated_duration": estimated_duration,
        "student_type": student_type,
        "document_complete": document_complete,
        "special_assistance": special_assistance
    }
    st.session_state.manual_records.append(new_record)
    st.success(f"Service request #{next_id} added!")

# --- 5. BUILD QUEUE & DISPLAY RESULTS ---
st.header("3. Queue Results")

if original is not None:
    working_data = original.copy()
else:
    working_data = pd.DataFrame(columns=required_columns)

if st.session_state.manual_records:
    manual_df = pd.DataFrame(st.session_state.manual_records)
    working_data = pd.concat([working_data, manual_df], ignore_index=True)

if working_data.empty:
    st.info("Upload your original dataset or add a service request to see the queue results.")
else:
    scored = score_records(working_data)
    priority_order, fifo_order, comparison = rank_records(scored)

    metric1, metric2, metric3 = st.columns(3)
    metric1.metric("Total Requests", len(scored))
    metric2.metric("High-Urgency Requests", int((scored["urgency"] == "High").sum()))
    metric3.metric("Average Priority Score", round(scored["priority_score"].mean(), 2))

    tab1, tab2, tab3, tab4 = st.tabs(["Priority Scoring", "FIFO", "Comparison", "Tests"])

    with tab1:
        st.subheader("Priority Scoring Results")
        st.caption("Higher priority scores are served first.")
        st.dataframe(priority_order, use_container_width=True, hide_index=True)

    with tab2:
        st.subheader("First In, First Out")
        st.caption("Earlier arrival times are served first.")
        st.dataframe(fifo_order, use_container_width=True, hide_index=True)

    with tab3:
        st.subheader("Priority Scoring vs. FIFO")
        st.dataframe(comparison.sort_values("priority_rank"), use_container_width=True, hide_index=True)

        top10 = comparison.sort_values("priority_rank").head(10)
        if not top10.empty:
            fig, ax = plt.subplots(figsize=(10, 5))
            x = np.arange(len(top10))
            width = 0.4
            ax.bar(x - width/2, top10["priority_rank"], width, label="Priority Scoring")
            ax.bar(x + width/2, top10["fifo_rank"], width, label="FIFO")
            ax.set_xticks(x)
            ax.set_xticklabels(top10["record_id"].astype(str), rotation=45)
            ax.invert_yaxis()
            ax.set_xlabel("Record ID")
            ax.set_ylabel("Rank (1 = served first)")
            ax.set_title("Priority Scoring versus FIFO")
            ax.legend()
            fig.tight_layout()
            st.pyplot(fig)
            plt.close(fig)

    with tab4:
        st.subheader("Boundary Test")
        boundary = pd.DataFrame({"estimated_duration": [14, 15, 16]})
        boundary["duration_score"] = np.where(boundary["estimated_duration"] <= 15, 5, 0)
        st.dataframe(boundary, hide_index=True)

        st.subheader("Equal-Score Test")
        same_score = priority_order[priority_order.duplicated("priority_score", keep=False)]
        st.dataframe(
            same_score[["record_id", "arrival_time", "priority_score", "priority_rank"]].head(10),
            use_container_width=True, hide_index=True
        )

        st.subheader("Missing-Value Test")
        if missing is not None:
            st.write("Missing values before handling:")
            st.dataframe(missing.isna().sum().rename("Missing Count"), use_container_width=True)
            missing_result = score_records(missing)
            st.write("Results after handling missing values:")
            st.dataframe(missing_result.head(), use_container_width=True, hide_index=True)
        else:
            st.info("Upload the missing-value CSV to run this test.")

    # --- 6. DOWNLOAD RESULTS ---
    st.header("4. Download Results")
    files_to_download = {
        "priority_scoring_results.csv": priority_order,
        "fifo_results.csv": fifo_order,
        "algorithm_comparison_all_records.csv": comparison,
        "algorithm_comparison_top10.csv": comparison.sort_values("priority_rank").head(10)
    }

    if missing is not None:
        files_to_download["missing_value_test_results.csv"] = score_records(missing)

    boundary = pd.DataFrame({"estimated_duration": [14, 15, 16]})
    boundary["duration_score"] = np.where(boundary["estimated_duration"] <= 15, 5, 0)
    files_to_download["boundary_test_results.csv"] = boundary

    col1, col2 = st.columns(2)
    with col1:
        for filename, df_download in files_to_download.items():
            st.download_button(
                label=f"Download {filename}",
                data=df_download.to_csv(index=False).encode("utf-8"),
                file_name=filename,
                mime="text/csv",
                key=f"download_{filename}",
                use_container_width=True
            )
    with col2:
        zip_buffer = io.BytesIO()
        with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zip_file:
            for filename, df_download in files_to_download.items():
                zip_file.writestr(filename, df_download.to_csv(index=False))
        st.download_button(
            label="Download All Results (ZIP)",
            data=zip_buffer.getvalue(),
            file_name="CampusServiceQueuePrioritization.zip",
            mime="application/zip",
            use_container_width=True
        )


Writing app.py


Run Application

In [3]:
!ls

app.py	sample_data


In [4]:
!ls /content/app.py

/content/app.py


In [9]:
!pkill -f streamlit

In [10]:
!streamlit run /content/app.py \
    --server.port 8502 \
    --server.address 0.0.0.0 \
    --server.headless true \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    --server.enableWebsocketCompression false \
    > /content/streamlit.log 2>&1 &

In [11]:
import time
from google.colab.output import eval_js

time.sleep(5)

app_url = eval_js("google.colab.kernel.proxyPort(8502)")
print("Open your Streamlit app:", app_url)

Open your Streamlit app: https://8502-m-s-kkb-usc1b1-1q1fmli5g56bc-b.us-central1-1.prod.colab.dev


In [8]:
!tail -n 40 /content/streamlit.log

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: No such option '--server.allowWebsocketCompression'. (Did you mean one of: '--server.allowedHosts', '--server.enableWebsocketCompression', '--server.websocketPingInterval'?)
